# PyDBAdminKit 0.5.x — Operations Lab

Notebook d’expérimentation de la ligne **Operations** de PyDBAdminKit, testé avec `0.5.0b1`.

Ce lab couvre :
- découverte des capabilities Backup / Restore / Maintenance ;
- `backup create` et `backup validate` ;
- planification et exécution optionnelle d’un restore vers une nouvelle base ;
- `VACUUM`, `ANALYZE`, `REINDEX` ;
- vues de progression VACUUM / REINDEX ;
- garde-fous `dry-run`, environnement et `read_only`.

> Les opérations réelles sont **désactivées par défaut**. Aucun secret n’est stocké dans ce notebook.


In [1]:
import os
import tempfile
from pathlib import Path
from uuid import uuid4

from pydbadminkit import __version__
from pydbadminkit.bootstrap import (
    build_backup_service,
    build_backup_validation_service,
    build_capability_service,
    build_catalog_service,
    build_maintenance_service,
    build_restore_service,
    resolve_connection,
)
from pydbadminkit.domain.common import parse_qualified_name
from pydbadminkit.domain.operations import (
    AnalyzeCommand,
    BackupFormat,
    CreateBackupCommand,
    ReindexCommand,
    ReindexTargetType,
    RestoreBackupCommand,
    VacuumCommand,
)
from pydbadminkit.domain.safety import MutationOptions

In [3]:
import os

os.environ["PYDBADMIN_NATIVE_PASSWORD"] = "postgres"
print(os.environ.get("PYDBADMIN_NATIVE_PASSWORD"))

postgres


In [4]:
CONNECTION_PROFILE = os.getenv("PYDBADMIN_PROFILE", "local-native")

RUN_BACKUP = os.getenv("PYDBADMIN_RUN_BACKUP", "0") == "1"
RUN_MAINTENANCE = os.getenv("PYDBADMIN_RUN_MAINTENANCE", "0") == "1"
RUN_RESTORE = os.getenv("PYDBADMIN_RUN_RESTORE", "0") == "1"

EXPLICIT_TABLE = os.getenv("PYDBADMIN_LAB_TABLE")
EXPLICIT_INDEX = os.getenv("PYDBADMIN_LAB_INDEX")
EXISTING_BACKUP = os.getenv("PYDBADMIN_LAB_BACKUP")
RESTORE_DATABASE = os.getenv(
    "PYDBADMIN_LAB_RESTORE_DB",
    f"pydbadmin_restore_lab_{uuid4().hex[:8]}",
)


def find_config_path() -> Path:
    configured = os.getenv("PYDBADMIN_CONFIG")
    if configured:
        return Path(configured).expanduser().resolve()

    for candidate in (
        Path.cwd() / "config.toml",
        Path.cwd().parent / "config.toml",
    ):
        if candidate.exists():
            return candidate.resolve()

    raise FileNotFoundError("config.toml introuvable. Définissez PYDBADMIN_CONFIG si nécessaire.")


def print_plan(plan) -> None:
    print("operation    :", plan.operation)
    print("target       :", plan.target)
    print("risk         :", plan.risk.label)
    print("confirmation :", plan.confirmation.value)
    for effect in plan.effects:
        print("effect       :", effect)
    for warning in plan.warnings:
        print("warning      :", warning)


CONFIG_PATH = find_config_path()
config = resolve_connection(CONNECTION_PROFILE, CONFIG_PATH)
CAN_MUTATE = not config.read_only and str(config.environment) in {"development", "testing"}

print("Version      :", __version__)
print("Profil       :", CONNECTION_PROFILE)
print("Config       :", CONFIG_PATH)
print("Database     :", config.database)
print("Environment  :", config.environment)
print("Read-only    :", config.read_only)
print("RUN_BACKUP   :", RUN_BACKUP)
print("RUN_MAINT.   :", RUN_MAINTENANCE)
print("RUN_RESTORE  :", RUN_RESTORE)

Version      : 0.5.0b1
Profil       : local-native
Config       : C:\Users\awounfouet\Projects\packages\pydbadminkit\config.toml
Database     : pydbadmin_dev
Environment  : development
Read-only    : False
RUN_BACKUP   : False
RUN_MAINT.   : False
RUN_RESTORE  : False


## 1. Capabilities Operations


In [5]:
capabilities = build_capability_service().list()
prefixes = ("backup.", "maintenance.", "postgres.")

for capability in capabilities:
    if capability.name.startswith(prefixes):
        reason = f" — {capability.reason}" if capability.reason else ""
        print(f"{capability.name:<36} {capability.availability.value}{reason}")

backup.create                        unavailable_tool — Required tool 'pg_dump' was not found.
backup.restore                       unavailable_tool — Required restore tool(s) not found: pg_restore, psql.
backup.validate                      available
maintenance.analyze                  available
maintenance.reindex                  available
maintenance.vacuum                   available
postgres.reindex.concurrently        available
postgres.reindex.progress            available
postgres.vacuum.progress             available


## 2. Cibles de maintenance et progress views


In [6]:
catalog = build_catalog_service(CONNECTION_PROFILE, CONFIG_PATH)
tables = catalog.list_tables(schema="public")
indexes = catalog.list_indexes(schema="public")

print(f"Tables public : {len(tables)}")
for table in tables[:10]:
    print(" -", table.name)

print(f"\nIndex public : {len(indexes)}")
for index in indexes[:10]:
    print(" -", index.name, "table=", index.table)

Tables public : 0

Index public : 0


In [7]:
maintenance = build_maintenance_service(
    CONNECTION_PROFILE,
    CONFIG_PATH,
)

vacuum_progress = maintenance.list_vacuum_progress()
reindex_progress = maintenance.list_reindex_progress()

print("VACUUM en cours :", len(vacuum_progress))
for item in vacuum_progress:
    print(item)

print("\nREINDEX en cours:", len(reindex_progress))
for item in reindex_progress:
    print(item)

VACUUM en cours : 0

REINDEX en cours: 0


## 3. Maintenance — dry-run

Le notebook choisit une table/index disponible uniquement pour le **dry-run**. Pour une exécution réelle, définissez explicitement `PYDBADMIN_LAB_TABLE` et éventuellement `PYDBADMIN_LAB_INDEX`.


In [8]:
dry_table = (
    parse_qualified_name(EXPLICIT_TABLE) if EXPLICIT_TABLE else (tables[0].name if tables else None)
)
dry_index = (
    parse_qualified_name(EXPLICIT_INDEX)
    if EXPLICIT_INDEX
    else (indexes[0].name if indexes else None)
)

print("Table dry-run :", dry_table)
print("Index dry-run :", dry_index)

Table dry-run : None
Index dry-run : None


In [9]:
if dry_table is None:
    print("[SKIP] Aucune table public disponible.")
else:
    vacuum_command = VacuumCommand(
        table=dry_table,
        analyze=True,
        statement_timeout_seconds=60,
        lock_timeout_seconds=5,
    )
    vacuum_plan = maintenance.plan_vacuum(vacuum_command)
    print_plan(vacuum_plan)
    maintenance.vacuum(
        vacuum_command,
        MutationOptions(dry_run=True),
        plan=vacuum_plan,
    )

[SKIP] Aucune table public disponible.


In [10]:
if dry_table is None:
    print("[SKIP] Aucune table public disponible.")
else:
    analyze_command = AnalyzeCommand(table=dry_table)
    analyze_plan = maintenance.plan_analyze(analyze_command)
    print_plan(analyze_plan)
    maintenance.analyze(
        analyze_command,
        MutationOptions(dry_run=True),
        plan=analyze_plan,
    )

[SKIP] Aucune table public disponible.


In [11]:
if dry_index is None:
    print("[SKIP] Aucun index public disponible.")
else:
    reindex_command = ReindexCommand(
        target_type=ReindexTargetType.INDEX,
        target=dry_index,
        concurrently=True,
        statement_timeout_seconds=120,
        lock_timeout_seconds=5,
    )
    reindex_plan = maintenance.plan_reindex(reindex_command)
    print_plan(reindex_plan)
    maintenance.reindex(
        reindex_command,
        MutationOptions(dry_run=True),
        plan=reindex_plan,
    )

[SKIP] Aucun index public disponible.


## 4. Backup — dry-run


In [12]:
LAB_DIR = Path(tempfile.gettempdir()) / "pydbadminkit-050-lab"
generated_backup = LAB_DIR / f"{config.database}_{uuid4().hex[:8]}.dump"
backup_path = Path(EXISTING_BACKUP).expanduser() if EXISTING_BACKUP else generated_backup

backup_service = build_backup_service(
    CONNECTION_PROFILE,
    CONFIG_PATH,
)
backup_command = CreateBackupCommand(
    database=config.database,
    format=BackupFormat.CUSTOM,
    output_path=str(backup_path),
    checksum=True,
    timeout_seconds=300,
)
backup_plan = backup_service.plan_create_backup(backup_command)

print_plan(backup_plan)

backup_service.create_backup(
    backup_command,
    MutationOptions(dry_run=True),
    plan=backup_plan,
)

backup_available = backup_path.exists()
print("Artefact existant :", backup_available)
print("Backup path       :", backup_path)

operation    : backup.create
target       : C:\Users\AWOUNF~1\AppData\Local\Temp\pydbadminkit-050-lab\pydbadmin_dev_2fc7195f.dump
risk         : low
confirmation : none
effect       : Create a custom logical backup of database 'pydbadmin_dev'.
effect       : Write backup artifact to 'C:\Users\AWOUNF~1\AppData\Local\Temp\pydbadminkit-050-lab\pydbadmin_dev_2fc7195f.dump'.
Artefact existant : False
Backup path       : C:\Users\AWOUNF~1\AppData\Local\Temp\pydbadminkit-050-lab\pydbadmin_dev_2fc7195f.dump


## 5. Backup réel optionnel + validation

Activez `PYDBADMIN_RUN_BACKUP=1` uniquement sur un profil `development/testing` avec `read_only=false`.


In [13]:
if not RUN_BACKUP:
    print("[SKIP] PYDBADMIN_RUN_BACKUP != 1")
elif not CAN_MUTATE:
    print("[SKIP] Profil non development/testing ou read_only=true.")
else:
    LAB_DIR.mkdir(parents=True, exist_ok=True)
    backup_result = backup_service.create_backup(
        backup_command,
        MutationOptions(approved=True),
        plan=backup_plan,
    )
    backup_available = True
    print("Backup ID :", backup_result.id)
    print("Path      :", backup_result.path)
    print("Size      :", backup_result.size_bytes)
    print("SHA-256   :", backup_result.checksum)

[SKIP] PYDBADMIN_RUN_BACKUP != 1


In [14]:
backup_available = backup_path.exists()

if not backup_available:
    print("[SKIP] Aucun artefact réel à valider.")
else:
    validation = build_backup_validation_service().validate_backup(str(backup_path))
    print("Valid    :", validation.valid)
    print("Level    :", validation.level)
    for warning in validation.warnings:
        print("Warning  :", warning)
    for error in validation.errors:
        print("Error    :", error)

[SKIP] Aucun artefact réel à valider.


## 6. Maintenance réelle optionnelle

L’exécution réelle n’utilise **jamais** automatiquement la première table trouvée. Définissez `PYDBADMIN_LAB_TABLE` pour choisir explicitement la cible.


In [15]:
if not RUN_MAINTENANCE:
    print("[SKIP] PYDBADMIN_RUN_MAINTENANCE != 1")
elif not CAN_MUTATE:
    print("[SKIP] Profil non development/testing ou read_only=true.")
elif not EXPLICIT_TABLE:
    print("[SKIP] Définissez PYDBADMIN_LAB_TABLE.")
else:
    table_target = parse_qualified_name(EXPLICIT_TABLE)

    vacuum_command = VacuumCommand(
        table=table_target,
        analyze=True,
        statement_timeout_seconds=60,
        lock_timeout_seconds=5,
    )
    vacuum_plan = maintenance.plan_vacuum(vacuum_command)
    vacuum_result = maintenance.vacuum(
        vacuum_command,
        MutationOptions(approved=True),
        plan=vacuum_plan,
    )
    print("VACUUM :", vacuum_result.status.value, vacuum_result.message)

    analyze_command = AnalyzeCommand(table=table_target)
    analyze_plan = maintenance.plan_analyze(analyze_command)
    analyze_result = maintenance.analyze(
        analyze_command,
        MutationOptions(approved=True),
        plan=analyze_plan,
    )
    print("ANALYZE:", analyze_result.status.value, analyze_result.message)

[SKIP] PYDBADMIN_RUN_MAINTENANCE != 1


In [16]:
if not RUN_MAINTENANCE:
    print("[SKIP] Maintenance réelle désactivée.")
elif not CAN_MUTATE:
    print("[SKIP] Profil non development/testing ou read_only=true.")
elif not EXPLICIT_INDEX:
    print("[SKIP] Définissez PYDBADMIN_LAB_INDEX pour tester REINDEX.")
else:
    index_target = parse_qualified_name(EXPLICIT_INDEX)
    reindex_command = ReindexCommand(
        target_type=ReindexTargetType.INDEX,
        target=index_target,
        concurrently=True,
        statement_timeout_seconds=120,
        lock_timeout_seconds=5,
    )
    reindex_plan = maintenance.plan_reindex(reindex_command)
    reindex_result = maintenance.reindex(
        reindex_command,
        MutationOptions(approved=True),
        plan=reindex_plan,
    )
    print("REINDEX:", reindex_result.status.value, reindex_result.message)

[SKIP] Maintenance réelle désactivée.


## 7. Restore — dry-run puis exécution optionnelle

Un artefact réel est nécessaire pour le preflight Restore. Par défaut, la cible est un nom de base unique et `create=True`.

> Le notebook ne supprime pas automatiquement la base restaurée : elle reste disponible pour inspection.


In [17]:
backup_available = backup_path.exists()

if not backup_available:
    print("[SKIP] Un artefact backup réel est requis.")
else:
    restore_service = build_restore_service(
        CONNECTION_PROFILE,
        CONFIG_PATH,
    )
    restore_command = RestoreBackupCommand(
        backup_path=str(backup_path),
        target_database=RESTORE_DATABASE,
        create=True,
        timeout_seconds=300,
    )
    restore_plan = restore_service.plan_restore(restore_command)
    print_plan(restore_plan)

    restore_service.restore(
        restore_command,
        MutationOptions(dry_run=True),
        plan=restore_plan,
    )

[SKIP] Un artefact backup réel est requis.


In [18]:
if not backup_available:
    print("[SKIP] Aucun backup réel.")
elif not RUN_RESTORE:
    print("[SKIP] PYDBADMIN_RUN_RESTORE != 1")
elif not CAN_MUTATE:
    print("[SKIP] Profil non development/testing ou read_only=true.")
else:
    restore_result = restore_service.restore(
        restore_command,
        MutationOptions(approved=True),
        plan=restore_plan,
    )
    print("Target       :", restore_result.target_database)
    print("Status       :", restore_result.status.value)
    print("Tool         :", restore_result.tool)
    print("Verification :", restore_result.verification_passed)
    print("La base restaurée est conservée pour inspection.")

[SKIP] Aucun backup réel.


## 8. Résultat attendu

À la fin de ce lab, vous devez pouvoir valider séparément :

| Domaine | Validation |
|---|---|
| Capabilities | Backup / Restore / Maintenance visibles |
| Progress | VACUUM / REINDEX interrogeables sans mutation |
| Maintenance | plans VACUUM / ANALYZE / REINDEX |
| Backup | plan, création opt-in, sidecar + SHA-256 |
| Restore | preflight, plan, création opt-in de cible |
| Guardrails | aucune mutation réelle sans opt-in explicite |

Les sorties des cellules ne sont pas versionnées.
